In [ ]:
import pandas as pd

df_exploration = pd.read_csv('../new_dataset.csv')


pd.set_option('display.max_columns', None)
df_exploration.head()

,id,chords,release_date,genres,decade,rock_genre,artist_id,main_genre,spotify_song_id,spotify_artist_id,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,50,<intro_1> Db Gb Db Eb7 Ab7 Db Ab7 Db Ab7 Db Gb...,2015-07-23,'folklore argentino' 'rock nacional' 'tango' '...,2010.0,rock nacional,artist_39,country,7asFHR74hcfNANNJ79cqVw,6ULt24uY5oov72S2fY1qoi,108293,7asFHR74hcfNANNJ79cqVw,Edmundo Rivero;Orquesta de Horacio Salgán,Great Voices of Tango,Al Mundo Le Falta un Tornillo,21,159373,False,0.554,0.487,1,-8.078,1,0.0620,0.9080,0.000000,0.0633,0.773,120.667,4,tango
1,64,<verse_1> G Emin G C D G Emin C G C D <chorus_...,1973-03-28,'album rock' 'classic rock' 'hard rock' 'rock',1970.0,hard rock,artist_49,rock,2eDdFHgqNJltzlvlZFVDWd,36QJpDe2go2KgaRleHCDTp,47718,2eDdFHgqNJltzlvlZFVDWd,Led Zeppelin,Houses of the Holy (Remaster),Over the Hills and Far Away - Remaster,64,290933,False,0.402,0.691,7,-10.274,1,0.0691,0.0679,0.133000,0.0955,0.206,96.950,4,hard-rock
2,133,<intro_1> D <verse_1> G D C D C D <verse_2> G ...,1996,'cantautor' 'latin afrobeat' 'latin alternativ...,1990.0,latin rock,artist_102,alternative,23P8ZlzwSC9Fi9xIMUf4Qh,4ssUf5gLb1GBLxi1BhPrVt,1329,23P8ZlzwSC9Fi9xIMUf4Qh,Jorge Drexler,Vaivén,Cerca del mar,23,222040,False,0.749,0.391,7,-16.177,1,0.0767,0.6030,0.000004,0.1160,0.351,96.035,4,afrobeat
3,261,<intro_1> Dmin C G Dmin C G <verse_1> Dmin C G...,2022-05-27,'ambient folk',2020.0,NaN,artist_150,country,6qr4bldy3y451hApJFEWqa,0sYlth2PW1zWJMEU2vCukz,4578,6qr4bldy3y451hApJFEWqa,Flower Face,The Shark in Your Water,Sugar Water,46,198592,False,0.487,0.604,0,-5.140,1,0.0319,0.6010,0.000000,0.1440,0.827,162.973,4,ambient
4,278,<intro_1> Fmin Gmin Dmin C Bb F Gmin Dmin Gmin...,2015-01-01,'bluegrass' 'modern southern rock' 'neo-tradit...,2010.0,modern southern rock,artist_166,rock,6qtbJulxYHxmoAtWz0Qnhs,3bLSAQPeix7Xm2e5Gtn48R,7092,6qtbJulxYHxmoAtWz0Qnhs,The Steeldrivers,The Muscle Shoals Recordings,Here She Goes,30,205720,False,0.330,0.331,5,-12.208,1,0.0322,0.6000,0.003210,0.1260,0.425,173.230,4,bluegrass


In [46]:
import re
import pandas as pd
import plotly.express as px

BIN = 1          # round tempo to the nearest multiple of this (1 = nearest BPM; try 5 or 10 if sparse)
TOP = 15         # number of most common chords to show
MIN_SONGS = 20   # drop genre/tempo groups with fewer songs than this

d = df_exploration.dropna(subset=['main_genre', 'tempo', 'chords']).copy()
d['tempo_r'] = ((d['tempo'] / BIN).round() * BIN).astype(int)

# Unique chords per song (section tags like <verse_1> removed)
def chords_in(s):
    return {t for t in str(s).split() if not re.match(r'<.*>', t)}

long = (d.assign(chord=d['chords'].apply(chords_in))
          .explode('chord')[['id', 'main_genre', 'tempo_r', 'chord']]
          .dropna())

top = long['chord'].value_counts().head(TOP).index
n = (d.groupby(['main_genre', 'tempo_r'])['id'].nunique()
       .rename('n_songs').reset_index())

agg = (long[long['chord'].isin(top)]
       .groupby(['main_genre', 'tempo_r', 'chord']).size()
       .rename('songs').reset_index()
       .merge(n, on=['main_genre', 'tempo_r']))
agg['share'] = agg['songs'] / agg['n_songs']
agg = agg[agg['n_songs'] >= MIN_SONGS]

fig = px.scatter(agg.sort_values('tempo_r'), x='tempo_r', y='chord',
                 size='share', color='share', color_continuous_scale='Viridis',
                 animation_frame='main_genre',
                 hover_data=['songs', 'n_songs'],
                 range_x=[d['tempo_r'].min() - 5, d['tempo_r'].max() + 5],
                 range_color=[0, agg['share'].max()],
                 labels={'tempo_r': 'tempo (BPM)'},
                 title='Chord Distribution by Tempo and Genre')
fig.show()
fig.write_html("../visualizations/chord_tempo_scatter.html", include_plotlyjs="cdn")

In [45]:
import re
import pandas as pd
import plotly.express as px

TOP = 15
MIN_SONGS = 20

d = df_exploration.dropna(subset=['main_genre', 'tempo', 'chords']).copy()
labels = ['Slow (<90)', 'Medium (90-120)', 'Fast (120-150)', 'Very fast (150+)']
d['tempo_bucket'] = pd.cut(d['tempo'], bins=[0, 90, 120, 150, 1000],
                           labels=labels, right=False)

def chords_in(s):
    return {t for t in str(s).split() if not re.match(r'<.*>', t)}

long = (d.assign(chord=d['chords'].apply(chords_in))
          .explode('chord')[['id', 'main_genre', 'tempo_bucket', 'chord']]
          .dropna())

top = long['chord'].value_counts().head(TOP).index
n = (d.groupby(['main_genre', 'tempo_bucket'], observed=True)['id'].nunique()
       .rename('n_songs').reset_index())

agg = (long[long['chord'].isin(top)]
       .groupby(['main_genre', 'tempo_bucket', 'chord'], observed=True).size()
       .rename('songs').reset_index()
       .merge(n, on=['main_genre', 'tempo_bucket']))
agg['share'] = agg['songs'] / agg['n_songs']
agg = agg[agg['n_songs'] >= MIN_SONGS]

fig = px.density_heatmap(agg, x='tempo_bucket', y='chord', z='share',
                         histfunc='avg', animation_frame='main_genre',
                         text_auto='.0%', color_continuous_scale='Viridis',
                         range_color=[0, agg['share'].max()],
                         category_orders={'tempo_bucket': labels},
                         title='Chord Distribution by Tempo and Genre')
fig.show()
fig.write_html("../visualizations/density_heatmap.html", include_plotlyjs="cdn")

In [ ]:
d = df_exploration.dropna(subset=['main_genre', 'valence', 'mode']).copy()
d['mode'] = d['mode'].map({1: 'major', 0: 'minor'})

fig = px.violin(d, x='main_genre', y='valence', color='mode',
                box=True, points=False,
                title='Valence by Genre and Mode')
fig.show()
fig.write_html("../visualizations/valence_by_genre_mode.html", include_plotlyjs="cdn")